# 🌿 AI Crop Health — Plant Disease Model Training
### EfficientNetV2B0 + Transfer Learning + Mixup + TTA
**Target: 98–99%+ accuracy on PlantVillage 38 classes**

---
## Steps for Kaggle:
1. **Environment Setup**: Ensure your Kaggle Notebook is set to **GPU T4x2** or **P100** (Session options -> Accelerator).
2. **Dataset**: Click "Add Data" on the right panel, search for "New Plant Diseases Dataset" by vipoooool, and add it.
3. Run each cell top to bottom!

> ⚡ **Important:** The best dataset currently is the Augmented PlantVillage dataset with 87k+ images. We use EfficientNetV2B0 as it offers state-of-the-art accuracy with extreme efficiency.

In [ ]:
# CELL 1: Check GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow version:', tf.__version__)
print('GPUs available:', len(gpus))
for g in gpus:
    print(' ->', g)
if not gpus:
    print('WARNING: No GPU detected! Turn on GPU in Kaggle Session Options.')
else:
    print('GPU ready!')

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
# CELL 2: Install packages
!pip install -q --upgrade scikit-learn tensorflow matplotlib Pillow seaborn opencv-python-headless tf-keras-vis

In [ ]:
# CELL 3: Check Dataset
import os
KAGGLE_DS_PATH = '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'

if os.path.exists(KAGGLE_DS_PATH):
    print("Dataset found at:", KAGGLE_DS_PATH)
else:
    print("ERROR: Dataset not found!")
    print("Please click '+ Add Data' in Kaggle, search for 'new-plant-diseases-dataset' (by vipoooool) and add it.")

In [ ]:
# CELL 4: Verify dataset structure
import os

TRAIN_DIR = os.path.join(KAGGLE_DS_PATH, 'train')
VALID_DIR = os.path.join(KAGGLE_DS_PATH, 'valid')

if os.path.exists(TRAIN_DIR):
    classes = sorted(os.listdir(TRAIN_DIR))
    total_train = sum(len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in classes)
    total_valid = sum(len(os.listdir(os.path.join(VALID_DIR, c))) for c in classes)
    print('Dataset Summary:')
    print('  Train imgs: ' + str(total_train))
    print('  Valid imgs: ' + str(total_valid))
    print('  Classes   : ' + str(len(classes)))
else:
    print('ERROR: Could not find train/ folder.')

In [ ]:
# CELL 5: Save class_indices.json
import json, os

MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)

class_indices = {"0": "Apple___Apple_scab", "1": "Apple___Black_rot", "2": "Apple___Cedar_apple_rust", "3": "Apple___healthy", "4": "Blueberry___healthy", "5": "Cherry_(including_sour)___Powdery_mildew", "6": "Cherry_(including_sour)___healthy", "7": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "8": "Corn_(maize)___Common_rust_", "9": "Corn_(maize)___Northern_Leaf_Blight", "10": "Corn_(maize)___healthy", "11": "Grape___Black_rot", "12": "Grape___Esca_(Black_Measles)", "13": "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "14": "Grape___healthy", "15": "Orange___Haunglongbing_(Citrus_greening)", "16": "Peach___Bacterial_spot", "17": "Peach___healthy", "18": "Pepper,_bell___Bacterial_spot", "19": "Pepper,_bell___healthy", "20": "Potato___Early_blight", "21": "Potato___Late_blight", "22": "Potato___healthy", "23": "Raspberry___healthy", "24": "Soybean___healthy", "25": "Squash___Powdery_mildew", "26": "Strawberry___Leaf_scorch", "27": "Strawberry___healthy", "28": "Tomato___Bacterial_spot", "29": "Tomato___Early_blight", "30": "Tomato___Late_blight", "31": "Tomato___Leaf_Mold", "32": "Tomato___Septoria_leaf_spot", "33": "Tomato___Spider_mites Two-spotted_spider_mite", "34": "Tomato___Target_Spot", "35": "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "36": "Tomato___Tomato_mosaic_virus", "37": "Tomato___healthy"}

with open(os.path.join(MODELS_DIR, 'class_indices.json'), 'w') as f:
    json.dump(class_indices, f, indent=2)

print('class_indices.json saved')

In [ ]:
# CELL 6: Configuration and Build EfficientNetV2B0 Model
import os, json, math, random, warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
warnings.filterwarnings('ignore')

# ---- CONFIG ----
IMG_SIZE      = 224
BATCH_SIZE    = 32
PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 35
LABEL_SMOOTH  = 0.1
DROPOUT_RATE  = 0.3
L2_REG        = 1e-4
MIXUP_ALPHA   = 0.2
LR_PHASE1     = 1e-3
LR_PHASE2     = 1e-4
LR_MIN        = 1e-6
UNFREEZE_FROM = -50
SEED          = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- GPU + Mixed Precision ----
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    mixed_precision.set_global_policy('mixed_float16')

# ---- PATHS ----
MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)
MODEL_H5  = MODELS_DIR + '/plant_disease_prediction_model.h5'
BEST_CKPT = MODELS_DIR + '/best_checkpoint.h5'
CLASS_JSON = MODELS_DIR + '/class_indices.json'

# ---- DATA GENERATORS ----
# NOTE: EfficientNetV2 expects inputs in [0, 255] range. Do NOT rescale=1./255.
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)

valid_datagen = ImageDataGenerator() # No rescaling

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

valid_gen = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=SEED
)

NUM_CLASSES  = len(train_gen.class_indices)
STEPS_TRAIN  = math.ceil(train_gen.samples / BATCH_SIZE)
STEPS_VALID  = math.ceil(valid_gen.samples  / BATCH_SIZE)

# ---- SAVE CLASS INDICES ----
gen_indices = {str(v): k for k, v in train_gen.class_indices.items()}
with open(CLASS_JSON, 'w') as f:
    json.dump(gen_indices, f, indent=2)

# ---- CLASS WEIGHTS ----
counts = np.bincount(train_gen.classes)
total  = float(sum(counts))
n_cls  = len(counts)
class_weights = {i: total / (n_cls * max(c, 1)) for i, c in enumerate(counts)}

# ---- COSINE LR ----
def cosine_lr(epoch, total, lr_max, lr_min):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * epoch / total))

# ---- BUILD MODEL ----
backbone = EfficientNetV2B0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

backbone.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = backbone(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(256, activation='relu',
                 kernel_regularizer=keras.regularizers.l2(L2_REG))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DROPOUT_RATE / 2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

model = Model(inputs, outputs, name='plant_disease_efficientnetv2b0')
print('Model built successfully: EfficientNetV2B0')

In [ ]:
# CELL 7: PHASE 1 - Train head only (backbone frozen)
print("=" * 55)
print("PHASE 1: Head training - backbone frozen")
print("Epochs: " + str(PHASE1_EPOCHS))
print("=" * 55)

model.compile(
    optimizer=keras.optimizers.Adam(LR_PHASE1),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]
)

callbacks_p1 = [
    ModelCheckpoint(BEST_CKPT, monitor="val_accuracy",
                    save_best_only=True, verbose=1, mode="max"),
    EarlyStopping(monitor="val_accuracy", patience=7,
                  restore_best_weights=True, verbose=1, mode="max"),
    keras.callbacks.LearningRateScheduler(
        lambda ep: cosine_lr(ep, PHASE1_EPOCHS, LR_PHASE1, LR_MIN), verbose=0),
]

# Note: Using train_gen directly (stable, no TF compat issues)
# Augmentation is already applied by ImageDataGenerator above
history1 = model.fit(
    train_gen,
    steps_per_epoch=STEPS_TRAIN,
    validation_data=valid_gen,
    validation_steps=STEPS_VALID,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_p1,
    verbose=1
)

p1_best = max(history1.history.get("val_accuracy", [0]))
print("Phase 1 complete. Best val accuracy: " + str(round(p1_best * 100, 2)) + "%")


In [ ]:
# CELL 8: PHASE 2 - Fine-tune (last 100 backbone layers unfrozen)
print("=" * 55)
print("PHASE 2: Fine-tuning - last 100 backbone layers")
print("Epochs: " + str(PHASE2_EPOCHS))
print("=" * 55)

backbone.trainable = True
for layer in backbone.layers[:UNFREEZE_FROM]:
    layer.trainable = False
# Keep BatchNorm frozen for stable fine-tuning
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_p2 = sum(tf.size(w).numpy() for w in model.trainable_weights)
print("Trainable params: " + str(trainable_p2))

model.compile(
    optimizer=keras.optimizers.Adam(LR_PHASE2),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]
)

callbacks_p2 = [
    ModelCheckpoint(BEST_CKPT, monitor="val_accuracy",
                    save_best_only=True, verbose=1, mode="max"),
    EarlyStopping(monitor="val_accuracy", patience=10,
                  restore_best_weights=True, verbose=1, mode="max"),
    keras.callbacks.LearningRateScheduler(
        lambda ep: cosine_lr(ep, PHASE2_EPOCHS, LR_PHASE2, LR_MIN), verbose=0),
]

# Reset generator before phase 2
train_gen.reset()

history2 = model.fit(
    train_gen,
    steps_per_epoch=STEPS_TRAIN,
    validation_data=valid_gen,
    validation_steps=STEPS_VALID,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_p2,
    verbose=1
)

p2_best = max(history2.history.get("val_accuracy", [0]))
print("Phase 2 complete. Best val accuracy: " + str(round(p2_best * 100, 2)) + "%")


In [ ]:
# CELL 9: Load best checkpoint, evaluate, and plot
import matplotlib.pyplot as plt

print('Loading best checkpoint...')
model = keras.models.load_model(BEST_CKPT)

valid_gen.reset()
results = model.evaluate(valid_gen, steps=STEPS_VALID, verbose=1)
final_loss = results[0]
final_acc  = results[1]
final_top5 = results[2]

print('=' * 50)
print('FINAL RESULTS')
print('Val Accuracy  : ' + str(round(final_acc * 100, 2)) + '%')
print('Top-5 Accuracy: ' + str(round(final_top5 * 100, 2)) + '%')
print('Val Loss      : ' + str(round(final_loss, 4)))
print('=' * 50)

# Training history plot
all_acc      = history1.history['accuracy']     + history2.history['accuracy']
all_val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss     = history1.history['loss']         + history2.history['loss']
all_val_loss = history1.history['val_loss']     + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
title = 'EfficientNetV2B0 — Val Accuracy: ' + str(round(final_acc * 100, 2)) + '%'
fig.suptitle(title, fontsize=13)
ep = range(1, len(all_acc) + 1)
ax1.plot(ep, [a * 100 for a in all_acc],     'b-',  label='Train')
ax1.plot(ep, [a * 100 for a in all_val_acc], 'r--', label='Val')
ax1.axvline(x=PHASE1_EPOCHS, color='gray', linestyle=':')
ax1.set_title('Accuracy (%)')
ax1.legend()
ax1.grid(alpha=0.3)
ax2.plot(ep, all_loss,     'b-',  label='Train')
ax2.plot(ep, all_val_loss, 'r--', label='Val')
ax2.axvline(x=PHASE1_EPOCHS, color='gray', linestyle=':')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plot_path = MODELS_DIR + '/training_history.png'
plt.savefig(plot_path, dpi=150)
plt.show()

In [ ]:
# CELL 10: Enterprise Evaluation - Advanced Metrics
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("Generating predictions for full validation set...")
valid_gen.reset()
Y_pred = model.predict(valid_gen, steps=STEPS_VALID, verbose=1)
y_pred = np.argmax(Y_pred, axis=1)
y_true = valid_gen.classes

class_labels = list(valid_gen.class_indices.keys())

print('\n' + '='*50)
print('CLASSIFICATION REPORT')
print('='*50)
report = classification_report(y_true, y_pred, target_names=class_labels, digits=4)
print(report)

print('\n' + '='*50)
print('CONFUSION MATRIX')
print('='*50)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, cmap='Blues', fmt='g', xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
cm_path = MODELS_DIR + '/confusion_matrix.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Confusion matrix saved to {cm_path}')

In [ ]:
# CELL 11: Enterprise Evaluation - Error Analysis
print('Analyzing False Positives / False Negatives (Difficult Cases)...')

incorrect_indices = np.where(y_pred != y_true)[0]
if len(incorrect_indices) == 0:
    print("Zero errors on validation set! Amazing.")
else:
    print(f"Total incorrectly classified images: {len(incorrect_indices)}")
    
    # Plot up to 9 incorrect predictions
    n_plot = min(9, len(incorrect_indices))
    sample_indices = np.random.choice(incorrect_indices, n_plot, replace=False)
    
    plt.figure(figsize=(15, 15))
    valid_gen.reset()
    
    batch_size = valid_gen.batch_size
    for i, idx in enumerate(sample_indices):
        batch_idx = idx // batch_size
        img_idx = idx % batch_size
        
        # Advance generator to correct batch
        valid_gen.reset()
        for _ in range(batch_idx):
            next(valid_gen)
        
        batch_x, batch_y = next(valid_gen)
        img = batch_x[img_idx]
        
        true_label = class_labels[y_true[idx]]
        pred_label = class_labels[y_pred[idx]]
        confidence = Y_pred[idx][y_pred[idx]] * 100
        
        plt.subplot(3, 3, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true_label[:20]}...\nPred: {pred_label[:20]}...\nConf: {confidence:.1f}%", 
                  color='red', fontsize=10)
        plt.axis('off')
        
    plt.tight_layout()
    error_path = MODELS_DIR + '/error_analysis.png'
    plt.savefig(error_path, dpi=150)
    plt.show()
    print(f'Error analysis plot saved to {error_path}')

In [ ]:
# CELL 12: Enterprise Evaluation - Explainability (Grad-CAM)
import cv2
from tensorflow.keras.models import Model
import tensorflow as tf

print('Generating Grad-CAM explanations to verify model focus...')

# Find the last convolutional layer in EfficientNetV2B0
last_conv_layer_name = None
for layer in reversed(model.layers[1].layers):  # model.layers[1] is the EfficientNet backbone
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

print(f"Using layer {last_conv_layer_name} for Grad-CAM")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Create a model that maps the input to the activations of the last conv layer + output predictions
    backbone = model.layers[1]
    grad_model = tf.keras.models.Model(
        [backbone.inputs], [backbone.get_layer(last_conv_layer_name).output, backbone.output]
    )
    
    # We then wrap it with the head of our main model
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = grad_model(inputs)[0] # activations
    
    # recreate head
    h = model.layers[2](x) # pooling
    for layer in model.layers[3:]:
        h = layer(h)
    
    full_grad_model = tf.keras.models.Model([inputs], [x, h])

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = full_grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Get a sample batch
valid_gen.reset()
sample_images, sample_labels = next(valid_gen)

plt.figure(figsize=(15, 10))
for i in range(min(4, len(sample_images))):
    img_array = np.expand_dims(sample_images[i], axis=0)
    
    # Predictions
    preds = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(preds[0])
    true_idx = np.argmax(sample_labels[i])
    
    # Generate Heatmap
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_idx)
    
    # Superimpose
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
    
    # Convert image back to 0-255 for cv2
    original_img = np.uint8(255 * sample_images[i])
    
    # overlay
    superimposed_img = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(sample_images[i])
    plt.title(f"Original\nTrue: {class_labels[true_idx][:15]}...")
    plt.axis('off')
    
    plt.subplot(2, 4, i + 5)
    plt.imshow(superimposed_img)
    plt.title(f"Grad-CAM\nPred: {class_labels[pred_idx][:15]}...")
    plt.axis('off')

plt.tight_layout()
gradcam_path = MODELS_DIR + '/gradcam_analysis.png'
plt.savefig(gradcam_path, dpi=150)
plt.show()
print(f'Grad-CAM analysis plot saved to {gradcam_path}')

# CELL 13: Enterprise Model Card

| Field | Description |
| :--- | :--- |
| **Model Architecture** | EfficientNetV2B0 (Transfer Learning, frozen backbone -> fine-tuned top layers) |
| **Intended Use** | Identification of 38 plant diseases and healthy crop variants from RGB images of leaves. |
| **Dataset Description** | Kaggle "New Plant Diseases Dataset (Augmented)" (~87,000 images, 80/20 train/valid split). |
| **Preprocessing** | No manual rescaling (EfficientNetV2 handles natively). Real-time augmentation: rotation (30°), zoom (0.2), shear (0.2), horizontal flip, brightness. |
| **Hyperparameters** | Optimizer: Adam, Learning Rate: Cosine Decay (1e-3 -> 1e-6), Label Smoothing: 0.1, L2 Reg: 1e-4, Batch Size: 32. |
| **Performance Checks** | ✅ Accuracy, ✅ Precision/Recall/F1, ✅ Confusion Matrix. |
| **Explainability Checks** | ✅ Grad-CAM (Heatmaps generated to verify localization). |
| **Error Analysis** | ✅ Difficult cases (False Positives/Negatives) manually visualized. |
| **Known Limitations** | Dataset relies entirely on laboratory-style images. Performance may degrade on in-field (wild) background noise unless further fine-tuned. |


In [ ]:
# CELL 14: Export .h5 and .tflite models

import numpy as np



# Save H5

model.save(MODEL_H5)

h5_size = os.path.getsize(MODEL_H5) / 1024 / 1024

print('H5 model saved: ' + MODEL_H5 + ' (' + str(round(h5_size, 1)) + ' MB)')



# TFLite float32

print('Converting to TFLite float32...')

conv_f32 = tf.lite.TFLiteConverter.from_keras_model(model)

conv_f32.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_f32_bytes = conv_f32.convert()

TFLITE_F32 = MODELS_DIR + '/plant_disease_model_float32.tflite'

with open(TFLITE_F32, 'wb') as f:

    f.write(tflite_f32_bytes)

f32_size = os.path.getsize(TFLITE_F32) / 1024 / 1024

print('Float32 TFLite saved: ' + TFLITE_F32 + ' (' + str(round(f32_size, 1)) + ' MB)')



# TFLite int8 quantized

print('Converting to TFLite int8 quantized...')

def rep_dataset():

    valid_gen.reset()

    count = 0

    for bx, _ in valid_gen:

        for img in bx:

            yield [np.expand_dims(img.astype(np.float32), 0)]

            count += 1

            if count >= 200:

                return



try:

    conv_int8 = tf.lite.TFLiteConverter.from_keras_model(model)

    conv_int8.optimizations = [tf.lite.Optimize.DEFAULT]

    conv_int8.representative_dataset = rep_dataset

    conv_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

    conv_int8.inference_input_type  = tf.uint8

    conv_int8.inference_output_type = tf.uint8

    tflite_int8_bytes = conv_int8.convert()

    TFLITE_INT8 = MODELS_DIR + '/plant_disease_model_int8.tflite'

    with open(TFLITE_INT8, 'wb') as f:

        f.write(tflite_int8_bytes)

    int8_size = os.path.getsize(TFLITE_INT8) / 1024 / 1024

    print('Int8 TFLite saved: ' + TFLITE_INT8 + ' (' + str(round(int8_size, 1)) + ' MB)')

except Exception as e:

    print('Int8 conversion skipped: ' + str(e))



print('')

print('All files saved in: ' + MODELS_DIR)

print('  plant_disease_prediction_model.h5')

print('  plant_disease_model_float32.tflite')

print('  class_indices.json')


In [ ]:
# CELL 15: Export Information

print('Training complete and models exported successfully.')

print('You can download the following files from the Kaggle Output pane (usually on the right side under "Data" -> "Output"):')

print(f'1. {MODEL_H5} (HDF5 format)')

print(f'2. {MODELS_DIR}/plant_disease_model_float32.tflite (TFLite Float32)')

try:

    print(f'3. {MODELS_DIR}/plant_disease_model_int8.tflite (TFLite Int8 Quantized)')

except:

    pass

print(f'4. {CLASS_JSON} (Class Indices mapping)')